# MEGA → Google Drive Importer для Google Colab

Один Colab-ноутбук для импорта публичных MEGA-ссылок в Google Drive.

### Что умеет эта версия
- публичные `mega.nz/file/...` и `mega.nz/folder/...`;
- несколько ссылок в очереди;
- сохранение структуры папок;
- отображение квоты Google Drive;
- предварительная проверка свободного места;
- загрузка в Drive через resumable upload;
- автоматический retry ошибок до 3 раз;
- пропуск уже существующего файла при совпадении имени и размера;
- прогресс, скорость и логи;
- кнопки запуска/остановки очереди;
- **встроенная поддержка 3-х видов веб-туннелей (Colab, Cloudflare, Localtunnel)**;
- **сохранение очереди и состояния между сессиями на Google Диске**;
- **аккуратный интерфейс со скрытым кодом**;
- **навигатор по папкам Google Drive прямо в интерфейсе**.

> ✅ **Важное улучшение:** База данных очереди теперь хранится в вашем Google Диске (в папке `MegaImporter_State`). Даже если сессия Colab прервется или вы перезагрузите страницу, при следующем запуске вы не потеряете историю и статус задач.

### Запуск
1. Запустите первую ячейку установки.
2. Запустите вторую ячейку.
3. Авторизуйте Google Drive (потребуется два разрешения: для диска и для API).
4. Выберите одну из предложенных ссылок для открытия интерфейса.
5. Выберите папку сохранения через файловый менеджер, вставьте ссылки и запустите импорт.


In [ ]:
#@title 🛠️ Установка зависимостей (Нажмите Play) { display-mode: "form" }
# megatools (содержит megadl) — альтернативный клиент MEGA.
# Преимущества перед официальным MEGAcmd:
#   - Чёткий exit code и stderr-сообщение при исчерпании квоты ("Transfer limit exceeded")
#   - Процесс сам завершается с ошибкой — нет зависаний
#   - Простая установка через apt

!apt-get update -qq
!apt-get install -y -qq megatools curl ca-certificates python3-pip

import subprocess

# Проверяем установку
check = subprocess.run(
    ["bash", "-lc", "megadl --version"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
if check.returncode != 0:
    raise RuntimeError("megadl не установлен. Проверьте логи выше.")

# Python-зависимости
!pip -q install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib flask werkzeug psutil

print(check.stdout.strip())
print("✅ Установка успешно завершена! Переходите к следующей ячейке.")


In [ ]:
#@title 🚀 Запуск: скачать код и запустить { display-mode: "form" }

GITHUB_REPO = "Leizer-San/Mega-to-Gdrive"  # <- замените на свой репозиторий
BRANCH      = "main"

import subprocess, sys, shutil

# Удаляем старую копию кода (чтобы всегда тянуть актуальную версию)
shutil.rmtree("/content/app", ignore_errors=True)

# Клонируем репозиторий
subprocess.run(
    ["git", "clone", "--depth=1", "--branch", BRANCH,
     f"https://github.com/{GITHUB_REPO}.git", "/content/app"],
    check=True,
)

# Добавляем путь к пакету и запускаем
sys.path.insert(0, "/content/app")
from mega_importer.server import run
run()
